# APTOS 2019 Blindness Detection - Data Processing

**Purpose:** Apply image preprocessing and define the augmentation pipeline for model training.

**This notebook covers:**
1. Load the train/validation splits from the previous step
2. Apply **Ben Graham's preprocessing** (Gaussian blur + circular crop + contrast enhancement)
3. Resize all images to a consistent target size
4. Compute normalization statistics from **training set only** (leakage prevention)
5. Define **data augmentation** pipeline (applied only during training)
6. Save processed images to `data/processed/`

**Key Principle:** All statistics and transformations are derived from the training split only, then applied to validation/test. This prevents data leakage.

## 1.0 Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from PIL import Image
import shutil

import warnings
warnings.filterwarnings('ignore')

SEED = 77
np.random.seed(SEED)

print('Libraries loaded successfully.')

## 2.0 Define Paths & Load Split Data

In [ ]:
# Define project paths
BASE_DIR = r'C:\Users\Owent\Desktop\ODL_Official_assignment\dataScience_workflow'
RAW_DATA_DIR = os.path.join(BASE_DIR, 'data', 'raw')
PROCESSED_DATA_DIR = os.path.join(BASE_DIR, 'data', 'processed')
TRAIN_IMAGES_DIR = os.path.join(RAW_DATA_DIR, 'train_images')
TEST_IMAGES_DIR = os.path.join(RAW_DATA_DIR, 'test_images')

# Output directories for processed images
PROCESSED_TRAIN_DIR = os.path.join(PROCESSED_DATA_DIR, 'train_images')
PROCESSED_VAL_DIR = os.path.join(PROCESSED_DATA_DIR, 'val_images')
PROCESSED_TEST_DIR = os.path.join(PROCESSED_DATA_DIR, 'test_images')

os.makedirs(PROCESSED_TRAIN_DIR, exist_ok=True)
os.makedirs(PROCESSED_VAL_DIR, exist_ok=True)
os.makedirs(PROCESSED_TEST_DIR, exist_ok=True)

# Target image size
IMG_SIZE = 224  # 224x224 for EfficientNet/ResNet compatibility

# Load splits created in 02_Data_Cleaning_and_Split.ipynb
train_df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'train_split.csv'))
val_df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'val_split.csv'))
test_df = pd.read_csv(os.path.join(RAW_DATA_DIR, 'test.csv'))

# Add file paths
train_df['file_path'] = train_df['id_code'].apply(lambda x: os.path.join(TRAIN_IMAGES_DIR, f'{x}.png'))
val_df['file_path'] = val_df['id_code'].apply(lambda x: os.path.join(TRAIN_IMAGES_DIR, f'{x}.png'))
test_df['file_path'] = test_df['id_code'].apply(lambda x: os.path.join(TEST_IMAGES_DIR, f'{x}.png'))

print(f'Training samples:   {len(train_df)}')
print(f'Validation samples: {len(val_df)}')
print(f'Test samples:       {len(test_df)}')
print(f'Target image size:  {IMG_SIZE}x{IMG_SIZE}')

---
## 3.0 Ben Graham's Preprocessing

Ben Graham's method (winner of the 2015 Diabetic Retinopathy competition) applies:
1. **Circular cropping** — Remove black borders around the retinal image
2. **Gaussian blur subtraction** — Subtract a heavily blurred version to enhance local contrast
3. **Rescaling** — Normalize to a consistent intensity range

This technique highlights microaneurysms, hemorrhages, and exudates while reducing noise from uneven illumination.

**Formula:** `processed = addWeighted(image, 4, GaussianBlur(image), -4, 128)`

### 3.1 Define Preprocessing Functions

In [ ]:
def crop_image_from_gray(img, tol=7):
    """
    Crop the black borders from a retinal image.
    Finds the bounding box of non-black pixels and crops to that region.
    
    Parameters:
        img: Input image (BGR)
        tol: Tolerance threshold for what's considered 'black'
    Returns:
        Cropped image
    """
    if img.ndim == 2:
        mask = img > tol
        return img[np.ix_(mask.any(1), mask.any(0))]
    elif img.ndim == 3:
        gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        mask = gray_img > tol
        
        # Check if image is not entirely black
        if not mask.any():
            return img
        
        # Find bounding box of non-black region
        img1 = img[:, :, 0][np.ix_(mask.any(1), mask.any(0))]
        img2 = img[:, :, 1][np.ix_(mask.any(1), mask.any(0))]
        img3 = img[:, :, 2][np.ix_(mask.any(1), mask.any(0))]
        img = np.stack([img1, img2, img3], axis=-1)
    return img


def apply_ben_graham_preprocessing(img, sigmaX=10):
    """
    Apply Ben Graham's preprocessing pipeline:
    1. Crop black borders
    2. Resize to target size
    3. Apply Gaussian blur subtraction for contrast enhancement
    
    Parameters:
        img: Input image (BGR format from cv2.imread)
        sigmaX: Gaussian blur sigma (controls blur strength)
    Returns:
        Processed image (BGR, uint8)
    """
    # Step 1: Crop black borders
    img = crop_image_from_gray(img, tol=7)
    
    # Step 2: Resize to target dimensions
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    
    # Step 3: Ben Graham's contrast enhancement
    # Subtract a blurred version and add 128 to center intensities
    img = cv2.addWeighted(
        img, 4,                                    # Original image weighted by 4
        cv2.GaussianBlur(img, (0, 0), sigmaX),    # Blurred version
        -4,                                        # Subtract blurred (weight -4)
        128                                        # Add 128 to center the result
    )
    
    return img


print('Preprocessing functions defined.')

### 3.2 Demonstrate Preprocessing on Sample Images

In [ ]:
# Show before/after preprocessing on one image per class
fig, axes = plt.subplots(5, 2, figsize=(10, 25))

for class_id in range(5):
    sample = train_df[train_df['diagnosis'] == class_id].iloc[0]
    img_original = cv2.imread(sample['file_path'])
    img_processed = apply_ben_graham_preprocessing(img_original.copy(), sigmaX=10)
    
    # Convert BGR to RGB for display
    axes[class_id, 0].imshow(cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB))
    axes[class_id, 0].set_title(f'Original - Class {class_id}', fontsize=11)
    axes[class_id, 0].axis('off')
    
    axes[class_id, 1].imshow(cv2.cvtColor(img_processed, cv2.COLOR_BGR2RGB))
    axes[class_id, 1].set_title(f'Ben Graham Processed - Class {class_id}', fontsize=11)
    axes[class_id, 1].axis('off')

plt.suptitle('Before vs After: Ben Graham Preprocessing', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 4.0 Apply Preprocessing to All Images

Process all training, validation, and test images and save to the `data/processed/` directory. This ensures:
- Consistent preprocessing across all sets
- Faster training (no on-the-fly heavy preprocessing needed)
- Reproducible results

### 4.1 Process Training Images

In [ ]:
# Process and save all training images
print(f'Processing {len(train_df)} training images...')

for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc='Training'):
    img = cv2.imread(row['file_path'])
    if img is not None:
        processed = apply_ben_graham_preprocessing(img, sigmaX=10)
        save_path = os.path.join(PROCESSED_TRAIN_DIR, f"{row['id_code']}.png")
        cv2.imwrite(save_path, processed)

print(f'\n✅ Training images saved to: {PROCESSED_TRAIN_DIR}')

### 4.2 Process Validation Images

In [ ]:
# Process and save all validation images
print(f'Processing {len(val_df)} validation images...')

for idx, row in tqdm(val_df.iterrows(), total=len(val_df), desc='Validation'):
    img = cv2.imread(row['file_path'])
    if img is not None:
        processed = apply_ben_graham_preprocessing(img, sigmaX=10)
        save_path = os.path.join(PROCESSED_VAL_DIR, f"{row['id_code']}.png")
        cv2.imwrite(save_path, processed)

print(f'\n✅ Validation images saved to: {PROCESSED_VAL_DIR}')

### 4.3 Process Test Images

In [ ]:
# Process and save all test images
print(f'Processing {len(test_df)} test images...')

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Test'):
    img = cv2.imread(row['file_path'])
    if img is not None:
        processed = apply_ben_graham_preprocessing(img, sigmaX=10)
        save_path = os.path.join(PROCESSED_TEST_DIR, f"{row['id_code']}.png")
        cv2.imwrite(save_path, processed)

print(f'\n✅ Test images saved to: {PROCESSED_TEST_DIR}')

---
## 5.0 Compute Normalization Statistics (Training Set Only)

**Why from training set only?**
- Normalization mean and standard deviation must come exclusively from training data
- Using validation/test statistics would leak future information into the training pipeline
- These values will be applied identically to validation and test sets during model training

In [ ]:
# Compute channel-wise mean and std from TRAINING images only
# This is critical for preventing data leakage

print('Computing normalization statistics from training set...')

channel_sum = np.zeros(3)
channel_sum_sq = np.zeros(3)
pixel_count = 0

for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc='Computing stats'):
    img_path = os.path.join(PROCESSED_TRAIN_DIR, f"{row['id_code']}.png")
    img = cv2.imread(img_path)
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img.astype(np.float64) / 255.0  # Normalize to [0, 1]
        
        channel_sum += img.sum(axis=(0, 1))
        channel_sum_sq += (img ** 2).sum(axis=(0, 1))
        pixel_count += img.shape[0] * img.shape[1]

# Calculate mean and std
channel_mean = channel_sum / pixel_count
channel_std = np.sqrt(channel_sum_sq / pixel_count - channel_mean ** 2)

print(f'\n=== Normalization Statistics (Training Set) ===')
print(f'Mean (R, G, B): [{channel_mean[0]:.4f}, {channel_mean[1]:.4f}, {channel_mean[2]:.4f}]')
print(f'Std  (R, G, B): [{channel_std[0]:.4f}, {channel_std[1]:.4f}, {channel_std[2]:.4f}]')

# Save statistics for use in model training
norm_stats = pd.DataFrame({
    'channel': ['R', 'G', 'B'],
    'mean': channel_mean,
    'std': channel_std
})
norm_stats_path = os.path.join(PROCESSED_DATA_DIR, 'normalization_stats.csv')
norm_stats.to_csv(norm_stats_path, index=False)
print(f'\nSaved to: {norm_stats_path}')
print('These values will be used to normalize ALL sets (train, val, test) during model training.')

---
## 6.0 Data Augmentation Pipeline

**Why augmentation?**
- The dataset is small (3,662 training images) and heavily imbalanced
- Augmentation artificially increases dataset diversity
- Helps the model generalize to unseen retinal images with different orientations/lighting

**Important:** Augmentation is applied **only to training data** during model training (not to validation/test).

**Augmentation strategies for retinal images:**
- **Horizontal/Vertical flip** — Retinal images can appear from either eye
- **Random rotation** — Fundus camera angle varies
- **Brightness/Contrast jittering** — Compensates for lighting variation
- **Random zoom/crop** — Simulates different camera distances

We define the pipeline here but it will be applied **on-the-fly during training** (not saved to disk) to save storage.

### 6.1 Define Augmentation Transforms (Keras/TensorFlow)

In [ ]:
# Define augmentation pipeline using tf.keras preprocessing layers
# This will be applied ONLY to training data during model training

import tensorflow as tf
from tensorflow.keras import layers

# Training augmentation pipeline
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.2),           # +/- 20% of full rotation (72 degrees)
    layers.RandomZoom(0.15),              # +/- 15% zoom
    layers.RandomContrast(0.2),           # +/- 20% contrast variation
    layers.RandomBrightness(0.1),         # +/- 10% brightness variation
], name='data_augmentation')

print('Augmentation pipeline defined.')
print('\nTransforms applied during TRAINING only:')
print('  - Random horizontal & vertical flip')
print('  - Random rotation (±72°)')
print('  - Random zoom (±15%)')
print('  - Random contrast (±20%)')
print('  - Random brightness (±10%)')
print('\nValidation/Test: NO augmentation applied (only resize + normalize).')

### 6.2 Visualize Augmentation Examples

In [ ]:
# Visualize augmentation on a sample image
sample_path = os.path.join(PROCESSED_TRAIN_DIR, f"{train_df.iloc[0]['id_code']}.png")
sample_img = cv2.imread(sample_path)
sample_img = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)
sample_tensor = tf.expand_dims(sample_img, 0)  # Add batch dimension

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes[0, 0].imshow(sample_img)
axes[0, 0].set_title('Original', fontsize=11)
axes[0, 0].axis('off')

for i in range(1, 8):
    row = i // 4
    col = i % 4
    augmented = data_augmentation(sample_tensor, training=True)
    axes[row, col].imshow(augmented[0].numpy().astype('uint8'))
    axes[row, col].set_title(f'Augmented #{i}', fontsize=11)
    axes[row, col].axis('off')

plt.suptitle('Data Augmentation Examples (Applied Only During Training)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 7.0 Compute Class Weights for Imbalanced Data

Since the dataset is heavily imbalanced (Class 0 has ~9x more samples than Class 3), we compute class weights to penalize misclassification of minority classes more heavily during training.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Compute balanced class weights from training set
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2, 3, 4]),
    y=train_df['diagnosis'].values
)

class_weight_dict = {i: w for i, w in enumerate(class_weights)}

print('=== Class Weights (Balanced) ===')
class_names = {0: 'No DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferative DR'}
for cls, weight in class_weight_dict.items():
    print(f'  Class {cls} ({class_names[cls]:>16}): {weight:.4f}')

print('\nHigher weight = model penalized more for misclassifying that class.')
print('This compensates for the class imbalance without oversampling.')

# Save class weights for model training
weights_df = pd.DataFrame({
    'class': list(class_weight_dict.keys()),
    'weight': list(class_weight_dict.values())
})
weights_path = os.path.join(PROCESSED_DATA_DIR, 'class_weights.csv')
weights_df.to_csv(weights_path, index=False)
print(f'\nSaved to: {weights_path}')

---
## 8.0 Processing Summary

### What was accomplished:

| Step | Action | Output |
|------|--------|--------|
| 3.0 | Ben Graham's preprocessing | Cropped, contrast-enhanced images |
| 4.0 | Process & save all images | `data/processed/{train,val,test}_images/` |
| 5.0 | Compute normalization stats | `normalization_stats.csv` (from train only) |
| 6.0 | Define augmentation pipeline | On-the-fly transforms (train only) |
| 7.0 | Compute class weights | `class_weights.csv` |

### Data Leakage Prevention:
- ✅ Normalization mean/std computed from **training images only**
- ✅ Augmentation applied to **training images only**
- ✅ Class weights computed from **training labels only**
- ✅ Same preprocessing function applied consistently to all sets

### Files Generated:
```
data/processed/
├── train_split.csv              (from previous notebook)
├── val_split.csv                (from previous notebook)
├── normalization_stats.csv      (channel mean/std)
├── class_weights.csv            (balanced weights)
├── train_images/                (processed training images)
├── val_images/                  (processed validation images)
└── test_images/                 (processed test images)
```

### Next Step:
→ `notebooks/model_training/Model_Training.ipynb` — Build and train the classification model using transfer learning